In [ ]:
RANDOM_STATE= 42
BATCH_SIZE = 128
LR = 2e-4
EPOCHS = 100

# xtransformer Config
HIDDEN_DIM=300
DEPTH=4
HEADS=4
MAX_LEN=256

MAX_VOCAB=20000
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_ID = 0
UNK_ID = 1

# Dataset Config
NUM_CLASSES = 10
TARGET_CLASSES = ["earn","acq","money-fx","grain","crude","trade","interest","ship","wheat","corn"]

In [ ]:
!pip install x-transformers

In [ ]:
import re
import nltk
import random
import numpy as np
import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
import torch.nn as nn
from sklearn.metrics import f1_score
from gensim.models import Word2Vec
from collections import Counter
from x_transformers import TransformerWrapper, Decoder
import inspect
print(inspect.signature(Decoder.__init__))

# Setting as large the xtick and ytick font sizes in graphs

# plt.rcParams['xtick.labelsize'] = 'large'
# plt.rcParams['ytick.labelsize'] = 'large'

<a id='IMDB'></a>
# Reuters-21578 (Text Categorization) Dataset

In [ ]:
# Storing the csv file into a DataFrame "df"

df = pd.read_csv('/kaggle/input/datasets/thedevastator/uncovering-financial-insights-with-the-reuters-2/ModLewis_train.csv')
df

In [ ]:
# Filter for target classes and create label mapping
# topics column contains list of topics, we'll take the first one if multiple exist
df['topics'] = df['topics'].apply(lambda x: eval(x) if isinstance(x, str) else x)
df['first_topic'] = df['topics'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)

# Filter only rows with target classes and non-null text
df = df[df['first_topic'].isin(TARGET_CLASSES)].copy()
df = df[df['text'].notna()].copy()  # Remove rows with NaN text

# Create label encoding
label_to_id = {label: idx for idx, label in enumerate(TARGET_CLASSES)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

df['label'] = df['first_topic'].map(label_to_id)
df = df[['text', 'label', 'first_topic']].reset_index(drop=True)
df

<a id='preprocessing'></a>
# Data preprocessing
First, we use regular expressions to make the following transformations to the reviews:

- remove punctuation marks
- remove HTML tags
- remove URL's
- remove characters which are not letters or digits
- remove successive whitespaces
- convert the text to lower case
- strip whitespaces from the beginning and the end of the reviews

In [ ]:
# Storing in "before_process" a random example of text before preprocessing
# Defining and applying the function "process" performing the transformations of the text
# Storing in "after_process" the example of text after preprocessing

idx = random.randint(0, len(df)-1)
before_process = df.iloc[idx]['text']

def process(x):
    if not isinstance(x, str):
        return ""
    x = re.sub(r'[,\.!?:()"]', '', x)
    x = re.sub(r'<.*?>', ' ', x)
    x = re.sub(r'http\S+', ' ', x)
    x = re.sub(r'[^a-zA-Z0-9]', ' ', x)
    x = re.sub(r'\s+', ' ', x)
    return x.lower().strip()

df['text'] = df['text'].apply(lambda x: process(x))
after_process = df.iloc[idx]['text']
after_process

Next, we remove stopwords from the reviews using the [word_tokenize()](https://www.nltk.org/_modules/nltk/tokenize.html#word_tokenize) function from the [nltk.tokenize]((https://www.nltk.org/api/nltk.tokenize.html) package.

In [ ]:
# Storing in "sw_set" the set of English stopwords provided by nltk
# Defining and applying the function "sw_remove" which remove stopwords from text
# Storing in "after_removal" the example of text after removal of the stopwords

sw_set = set(nltk.corpus.stopwords.words('english'))

def tokenize(text):
    return nltk.tokenize.word_tokenize(text)
    
def sw_remove(x):
    words = tokenize(x.lower())
    filtered_list = [word for word in words if word not in sw_set]
    return filtered_list

df['text'] = df['text'].apply(lambda x: sw_remove(x))
after_removal = sw_remove(after_process)
after_removal

<a id='splitting'></a>
# Data splitting and tokenization
We start by splitting our DataFrame into a training and test lists. We use the [train_test_split()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) function from the [sklearn.model_selection](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.model_selection) module which allow to perform the splitting randomly with respect to the index of the DataFrame.

In [ ]:
from sklearn.model_selection import train_test_split

# df tokenized success with nltk
train_text, tmp_text, train_label, tmp_label = train_test_split(df['text'], df['label'], test_size=0.1, random_state=RANDOM_STATE, stratify=df['label'])
test_text, val_text, test_label, val_label = train_test_split(tmp_text, tmp_label, test_size=0.5, random_state=RANDOM_STATE, stratify=tmp_label)


print('\033[1m' + 'train_text:' + '\033[0m', train_text)
print('\033[1m' + 'train_text.shape:' + '\033[0m', train_text.shape)
print('\033[1m' + 'test_text.shape:' + '\033[0m', test_text.shape)
print('\033[1m' + 'val_label.shape:' + '\033[0m', val_label.shape)
print('\033[1m' + 'val_label:' + '\033[0m', val_label)

#**Text to Vector by word2vec**

In [ ]:
import multiprocessing
print("CPU cores:", multiprocessing.cpu_count())
CPU_CORES=4

In [ ]:
tokenized_train = train_text.tolist() # pandas Series -> list
vectorize_model = Word2Vec(
    sentences=tokenized_train, 
    vector_size=HIDDEN_DIM, # embedding size
    window=5,
    min_count=1,
    workers=CPU_CORES,
    sg=1 # 0 = CBOW, 1 = Skip-gram
)

print(type(tokenized_train))
print(type(tokenized_train[0]))
print(tokenized_train[0][:10])

# สร้าง embedding matrix ตาม vocab for initial weight

In [ ]:
# text to vector
X_train = train_text
X_val = val_text
X_test = test_text

y_train = (train_label).astype(np.int64).to_numpy()
y_val   = (val_label).astype(np.int64).to_numpy()
y_test  = (test_label).astype(np.int64).to_numpy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("=========================")
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
counter = Counter()
for t in train_text:
    counter.update(t)

vocab = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
for i, (w, _) in enumerate(counter.most_common(MAX_VOCAB - 2), start=2):
    vocab[w] = i

def encode(tokens_list):
    ids = [vocab.get(t, vocab[UNK_TOKEN]) for t in tokens_list][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

vocab_size = len(vocab) # 

# Create token_id

In [ ]:
embedding_weight = np.zeros((vocab_size, HIDDEN_DIM), dtype=np.float32)

# ตารางคำศัพท์ → เวกเตอร์
# PAD row (0) = 0 
# UNK row (1) สุ่มเล็กน้อย
rng = np.random.default_rng(RANDOM_STATE)
embedding_weight[UNK_ID] = rng.normal(0, 0.01, size=(HIDDEN_DIM,)).astype(np.float32)

for word, idx in vocab.items():
    if word in (PAD_TOKEN, UNK_TOKEN):
        continue
    if word in vectorize_model.wv:
        embedding_weight[idx] = vectorize_model.wv[word]

# X-transformer Part

In [ ]:
class ImdbDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.tolist()     # เช่น train_rev
        self.labels = labels         # numpy array y_train (0/1)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = encode(self.texts[idx])                 # list length = MAX_LEN
        x = torch.tensor(ids, dtype=torch.long)       # (T,)
        y = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return x, y


In [ ]:
class XTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_matrix, num_classes=NUM_CLASSES):
        super().__init__()
        # self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # if embedding_matrix is not None:
        #     self.embedding.weight.data.copy_(torch.tensor(embedding_matrix, dtype=torch.float32))
        # self.embedding.weight.requires_grad = True
        
        self.transformer = TransformerWrapper(
            num_tokens=vocab_size,
            max_seq_len=MAX_LEN,
            attn_layers=Decoder(
                dim=HIDDEN_DIM,
                depth=DEPTH,
                heads=HEADS,
            )
        )

        self.load_w2v_into_wrapper(self.transformer, embedding_weight=embedding_matrix)
        self.emb_dropout = nn.Dropout(0.2)
        self.cls_dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(HIDDEN_DIM, num_classes)
    
    def load_w2v_into_wrapper(self, wrapper, embedding_weight):
        W = torch.tensor(embedding_weight, dtype=torch.float32)
        
        for name, module in wrapper.named_modules():
            if isinstance(module, nn.Embedding) and module.weight.shape == W.shape:
                module.weight.data.copy_(W)
                module.weight.requires_grad = True
                module.weight.data[0].zero_()
                print("✅ Loaded w2v into:", name, module.weight.shape) # expect [vocab_size, embedding_dim]
                return
        
        raise RuntimeError("❌ not found nn.Embedding in TransformerWrapper for this w2v shape")

    def forward(self, input_ids):
        mask = (input_ids != 0)  # (B,T)  padding_idx=0
        emb = self.transformer(input_ids, mask=mask, return_embeddings=True)  # (B,T,D)
        emb = self.emb_dropout(emb)  # dropout บน token representations
        # print(type(emb), emb.shape)
        # print(self.transformer)
        mask = mask.unsqueeze(-1).float()  # (B,T,1)
        pooled = (emb * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)  # (B,D)
        pooled = self.cls_dropout(pooled)
        return self.classifier(pooled)



In [ ]:
model = XTransformerClassifier(vocab_size=vocab_size, embedding_matrix=embedding_weight).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total = 0.0, 0

    all_preds = []
    all_labels = []

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * y.size(0)
        total += y.size(0)

        preds = logits.argmax(dim=1)

        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(y.detach().cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)

    acc = (y_pred == y_true).mean()
    f1 = f1_score(y_true, y_pred, average="weighted")  # changed to weighted for multi-class
    
    return total_loss / total, acc, f1

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return running_loss / total, correct / total

In [ ]:
def fit(model, train_loader, val_loader, optimizer, criterion, device, epochs):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc*100:.2f}% | "
            f"val_loss={val_loss:.4f} val_acc={val_acc*100:.2f}% val_f1={val_f1:.4f}"
        )

    return history

In [ ]:
def test(model, test_loader, criterion, device):
    test_loss, test_acc, test_f1 = evaluate(model, test_loader, criterion, device)
    print(f"TEST | loss={test_loss:.4f} acc={test_acc*100:.2f}% f1={test_f1:.4f}")
    return test_loss, test_acc, test_f1

In [ ]:
train_ds = ImdbDataset(X_train, y_train)
val_ds   = ImdbDataset(X_val, y_val)
test_ds  = ImdbDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Train and Test

In [ ]:
history_ = fit(model, train_loader, val_loader, optimizer, criterion, DEVICE, epochs=EPOCHS)
test_loss, test_acc,test_f1 = test(model, test_loader, criterion, DEVICE)

In [ ]:
@torch.no_grad()
def predict_text(model, text, device):
    model.eval()

    # preprocess
    text = sw_remove(process(text))
    
    ids = encode(text)
    x = torch.tensor([ids], dtype=torch.long, device=device)  # (1, MAX_LEN)

    logits = model(x)
    probs = torch.softmax(logits, dim=1).squeeze(0)  # (NUM_CLASSES,)

    pred_id = int(torch.argmax(probs).item())
    confidence = float(probs[pred_id].item()) * 100

    label = id_to_label[pred_id]
    return label, confidence, probs.detach().cpu().numpy()

In [ ]:
label, conf, probs = predict_text(model,"Qtly div 35 cts vs 35 cts prior Payable March 31 Record March nine Reuter.", DEVICE)
print("result:",label, f"{conf:.8f}%")
print("All probabilities:", {id_to_label[i]: f"{probs[i]*100:.2f}%" for i in range(NUM_CLASSES)})